# Try out the NVIDIA AI-Q Blueprint!
This notebook deploys the NVIDIA AI-Q Blueprint — a reference application for an enterprise-grade research agent built with the [NeMo Agent toolkit](https://docs.nvidia.com/nemo/agent-toolkit/latest/). It uses a two-tier research architecture that keeps simple queries fast while reserving multi-phase deep research for complex topics. The blueprint demonstrates an AI-powered research workflow that combines intent-aware routing, shallow and deep research agents, and optional human-in-the-loop clarification, plus benchmarks and evaluation harnesses for evaluation-driven development.

Some of the key capabilities of this blueprint are:

<table>
  <thead>
    <tr><th>Capability</th><th>Description</th><th>Value</th></tr>
  </thead>
  <tbody>
    <tr><td>Intelligence Dynamic Routing</td><td>Selects the right model, workflow depth, and data sources</td><td>optimal cost & performance</td></tr>
    <tr><td>Custom Evaluation Metrics</td><td>Evaluation harness with transparent reasoning</td><td>trust, auditability, reliability</td></tr>
    <tr><td>Open Recipe</td><td>Fully open, modular codebase with pluggable components</td><td>easy to customize and extend</td></tr>
    <tr><td>Persistent Context Management</td><td>Virtual file system and workspaces for data, code, and insights</td><td>handles long-running workflows</td></tr>
  </tbody>
</table>

In this notebook, you will install the prerequisites, set up the workflow environment, and connect to NVIDIA-hosted NIM APIs. Once deployed, you will have a fully functional research assistant that can answer simple questions via fast tool-augmented lookup (shallow research) and produce citation-backed, publication-style reports for complex topics (deep research). You can run the blueprint with web search only, customize models and tools, and optionally add pluggable knowledge retrieval.


Note: this notebook is designed to run as a [brev.dev launchable](https://brev.nvidia.com/launchable/deploy?launchableID=env-3ArAhMZxDcUsqbBUL8SbE2fAbKp)

![Architecture diagram](https://github.com/NVIDIA-AI-Blueprints/aiq/raw/develop/docs/assets/AIQ-arch-light.png)

# Getting Started

> [Software Components](#software-components)  
> [Hardware Requirements](#hardware-requirements)  
> [Prerequisites](#prerequisites)  
> [Spin Up Blueprint](#spin-up-blueprint)  
> [Environment Setup](#environment-setup)  
> [Run agent with NAT](#run-this-agent-using-the-nemo-agent-toolkit-run-command)  
> [Meta / Shallow / Deep examples](#meta-responses)  
> [Deployment using Docker Compose](#deployment-using-docker-compose)  
> [Next Steps](#next-steps)

## Software Components

The following are used by this project:

- [NVIDIA NeMo Agent toolkit](https://docs.nvidia.com/nemo/agent-toolkit/latest/)
- [NIM of nvidia/nemotron-3-super-120b-a12b](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [NIM of nvidia/nemotron-3-nano-30b-a3b](https://build.nvidia.com/nvidia/nemotron-3-nano-30b-a3b)
- [NIM of nvidia/llama-nemotron-embed-vl-1b-v2](https://build.nvidia.com/nvidia/llama-nemotron-embed-vl-1b-v2) (Optional)
- [NIM of nvidia/nemotron-nano-12b-v2-vl](https://build.nvidia.com/nvidia/nemotron-nano-12b-v2-vl) (Optional)
- [NVIDIA nvidia/nemotron-mini-4b-instruct](https://build.nvidia.com/nvidia/nemotron-mini-4b-instruct) (Optional)
- [Tavily Search API](https://tavily.com/) for web search

## Hardware Requirements

This notebook is intended to be run in a Brev environment with GPU access for deploying Nemotron 3 Super.

If you are running in an environment without GPU access, use a different configuration file under the `configs/` directory and `docker-compose.yaml` instead of `docker-compose-brev.yaml`.

This notebook deploys `nvidia/nemotron-3-super-120b-a12b` locally using vLLM. See the [model card](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16) for GPU requirements.



## Prerequisites

- **Docker Compose**
- **LLM API key** – For your chosen provider (required for LLM inference):
    - **NVIDIA API Key** from [NVIDIA AI](https://build.nvidia.com/) (for NIM models)
Optional requirements:
- **Web search API key**:
  - **Tavily** – [tavily.com](https://tavily.com) (for general web search)
- **Node.js 22+ and npm** (optional, for web UI mode)

## Spin Up Blueprint

We will be using docker compose to deploy this blueprint and interact with the NAT `run` command as well as through the custom chat-based frontend UI.

### Environment Setup

1. **Clone** the NVIDIA AI-Q Blueprint workflow repository
>**Note**: Skip this step if you are not on brev and instead running from within the repository.

In [ ]:
%%bash
# Only run this cell in a brev launchable
git clone https://github.com/NVIDIA-AI-Blueprints/aiq.git

In [ ]:
# Only run this cell in a brev launchable
# moves the notebook into the directory
%cd /home/ubuntu/aiq
repo_root = "/home/ubuntu/aiq"

2. **Setup** Environment

In [ ]:
# Run setup from the project root (where pyproject.toml and scripts/setup.sh live).
import os
from pathlib import Path

cwd = Path.cwd().resolve()
configured_repo_root = globals().get("repo_root")
REPO_ROOT = Path(configured_repo_root).expanduser().resolve() if configured_repo_root else None

search_roots = [cwd, *cwd.parents]
if REPO_ROOT is not None:
    search_roots = [REPO_ROOT, *search_roots]

REPO_ROOT = None
for candidate in search_roots:
    if (candidate / "pyproject.toml").exists() and (candidate / "scripts" / "setup.sh").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing pyproject.toml and scripts/setup.sh")

os.environ["REPO_ROOT"] = str(REPO_ROOT)
os.chdir(REPO_ROOT)

3. **Obtain and set** API keys and environment variables

<table>
  <thead>
    <tr><th>API</th><th>Environment Variable</th><th>Purpose</th></tr>
  </thead>
  <tbody>
    <tr><td>NVIDIA</td><td><code>NVIDIA_API_KEY</code></td><td>NIM / NGC model inference (required)</td></tr>
    <tr><td>Tavily</td><td><code>TAVILY_API_KEY</code></td><td>Web search (recommended)</td></tr>
  </tbody>
</table>

**NVIDIA API key:**  
1. Go to [NVIDIA Build](https://build.nvidia.com/explore/discover) or [NGC](https://ngc.nvidia.com/).  
2. Sign in and open a model card (e.g. Nemotron).  
3. Use **Get API Key** → **Generate Key**, then copy the key (starts with `nvapi-`).  

**Tavily API key:**  
1. Go to [tavily.com](https://tavily.com) and create an account.  
2. Create an API key in the dashboard and add it to your environment.  

In [ ]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ or os.environ["NVIDIA_API_KEY"] == "":
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

if "TAVILY_API_KEY" not in os.environ or os.environ["TAVILY_API_KEY"] == "":
    tavily_api_key = getpass.getpass("Enter your Tavily API key: ")
    os.environ["TAVILY_API_KEY"] = tavily_api_key

### Deployment using Docker Compose

1. Set environment variables for docker compose deployment

In [ ]:
os.environ["BACKEND_URL"] = "http://aiq-agent:8000"
os.environ["REQUIRE_AUTH"] = "false"
os.environ["BACKEND_CONFIG"] = (
    "/app/configs/config_web_default_llamaindex_brev.yml"  # path to the NAT config file from inside the docker container
)

2. Create an env file with secrets

In [ ]:
%%bash
cat > deploy/.env <<EOF
APP_ENV=development
LOG_LEVEL=INFO
BACKEND_CONFIG=/app/configs/config_web_default_llamaindex_brev.yml

AIQ_DEV_ENV=cli

NVIDIA_API_KEY=${NVIDIA_API_KEY}
TAVILY_API_KEY=${TAVILY_API_KEY}
EOF

Or run `cp deploy/.env.example deploy/.env` and replace the placeholders in the created `.env` file

3. Deploy the containers

<div class="alert alert-block alert-info">
<b>Note:</b> The following cell can take several minutes to run in order to deploy a vLLM server.
</div>

In [ ]:
!docker compose -f deploy/compose/docker-compose-brev.yaml up -d --build

4. Verify the containers are running

In [ ]:
# Confirm all the below mentioned containers are running.
import subprocess

result = subprocess.run(
    ["docker", "ps", "--format", "table {{.ID}}\t{{.Names}}\t{{.Status}}"],
    check=False,
    capture_output=True,
    text=True,
)

print(result.stdout)

The output should look something like this:
<div class="alert alert-block alert-info">
<b>Note:</b> aiq-vllm needs to progress from "starting" to "healthy".
</div>

```
CONTAINER ID   NAMES              STATUS
07bc9ee9df3e   aiq-blueprint-ui   Up 20 minutes (healthy)
9be9d410c15d   aiq-agent          Up 20 minutes
af18876ed59c   aiq-postgres       Up 20 minutes (healthy)
dae9987234b7   aiq-vllm           Up 20 minutes (healthy)
```

At this point, you should be able to access the NVIDIA AI-Q frontend web application by visiting http://localhost:3000.

>**Tip:** If you are running this notebook on brev, you will need to make the port for the AI-Q frontend accessible. On the settings page for your machine, navigate to "Using Ports", enter "3000", click "Expose Port", and then click "I accept".

**Select data sources:** In the UI, open the **data sources** panel (e.g. from the right side or connections icon). You will see available sources (e.g. Web Search, and optionally Paper Search, Knowledge Layer, or enterprise sources if configured). Toggle **on** the sources you want the agent to use for the next query. Web Search is usually enabled by default.

**Send a query:** Type your question or research request in the chat input and send it. The agent will use only the **enabled** data sources for that query. For deep research, the UI may show clarification or plan approval steps; respond as prompted. Results and the final report will appear in the chat.

In [ ]:
# Bring docker containers down
!docker compose -f deploy/compose/docker-compose-brev.yaml down -v

In [ ]:
# Optional: Reset the environment
# use only if you want to stop containers, remove volumes, and delete cached images
!docker compose -f deploy/compose/docker-compose-brev.yaml down -v --rmi all

### Run this agent using the NeMo Agent Toolkit [`run` command](https://docs.nvidia.com/nemo/agent-toolkit/latest/run-workflows/about-running-workflows.html#using-the-nat-run-command)

In [ ]:
%%writefile config_simple_researcher.yml

general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO

llms:
  nemotron_llm_intent:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    temperature: 0.5
    top_p: 0.9
    max_tokens: 4096
    num_retries: 5
    chat_template_kwargs:
      enable_thinking: true

  nemotron_nano_llm:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    temperature: 0.1
    top_p: 0.3
    max_tokens: 16384
    num_retries: 5
    chat_template_kwargs:
      enable_thinking: true

  nemotron_super_local_llm:
    _type: nim
    model_name: nemotron-super
    base_url: "http://localhost:8080/v1"
    temperature: 1.0
    top_p: 1.0
    max_tokens: 65536
    num_retries: 5
    chat_template_kwargs:
      enable_thinking: true

  # Hosted Nemotron Super is compatible and tested with AIQ but has limited availability
  # on the Build API due to high demand.
  # Uncomment nemotron_super_llm below if the endpoint is accessible.
  # nemotron_super_llm:
  #   _type: nim
  #   model_name: nvidia/nemotron-3-super-120b-a12b
  #   base_url: "https://integrate.api.nvidia.com/v1"
  #   temperature: 1.0
  #   top_p: 1.0
  #   max_tokens: 128000
  #   num_retries: 5
  #   chat_template_kwargs:
  #     enable_thinking: true
  
  gpt_oss_llm:
    _type: nim
    model_name: openai/gpt-oss-120b
    base_url: https://integrate.api.nvidia.com/v1
    temperature: 1.0
    top_p: 1.0
    max_tokens: 256000
    api_key: ${NVIDIA_API_KEY}
    max_retries: 10

functions:
  web_search_tool:
    _type: tavily_web_search
    max_results: 5
    max_content_length: 1000

  advanced_web_search_tool:
    _type: tavily_web_search
    max_results: 2
    advanced_search: true

  intent_classifier:
    _type: intent_classifier
    llm: nemotron_llm_intent
    tools:
      - web_search_tool

  shallow_research_agent:
    _type: shallow_research_agent
    llm: nemotron_nano_llm
    tools:
      - web_search_tool

  deep_research_agent:
    _type: deep_research_agent
    orchestrator_llm: gpt_oss_llm
    planner_llm: gpt_oss_llm
    researcher_llm: nemotron_super_local_llm  # replace with nemotron_super_llm if available
    max_loops: 2
    tools:
      - advanced_web_search_tool

workflow:
  _type: chat_deepresearcher_agent
  interactive_auth: false
  checkpoint_db: ${AIQ_CHECKPOINT_DB:-./checkpoints.db}

**What this config does:** 

This config defines a NAT workflow for the AI-Q chat researcher. 

**`general`** turns on console logging. 

**`general.front_end`** wires the Web API (AI-Q API Worker), async job store, and CORS for the chat UI.

**`llms`** wires the models for the agents: Nemotron Nano for intent/orchestration/planner, GPT-OSS or similar for clarifier and deep-research orchestrator/planner, and Nemotron 3 Super for the deep research researcher, with different settings per role for best performance.

**`functions`** register tools (Tavily web search, plus advanced search), the intent classifier, the clarifier agent, and the shallow and deep research agents with their assigned tools and LLMs.
 
**`workflow`** is set to `chat_deepresearcher_agent`. See source code for this full agent [here](../../src/aiq_agent/agents/chat_researcher/agent.py).

**Optional settings:** 

`enable_clarifier: true` lets the agent ask clarifying questions before deep research; 
`enable_plan_approval: true` (on the clarifier) lets users approve or adjust the research plan. 

>**Note:** the above human-in-the-loop interactions may not work in a notebook environment; you can experience these using the AI-Q CLI or the web UI deployed later in this notebook.

**Main agents:** intent classifier, clarifier, shallow research, deep research. 
**Tools:** web search tool for shallow researcher, advanced web search for deep research, and knowledge retrieval (using the LlamaIndex backend for user-uploaded documents).

>**Optional — Frontier model config:** For improved deep research report quality, you can use `configs/config_frontier_models.yml`, that replaces GPT-OSS with a frontier model (e.g. GPT-5.2) for orchestration and planning. When using Docker Compose or the web UI, set `BACKEND_CONFIG` to `/app/configs/config_frontier_models.yml`. Remember to set the required API key in your `.env` file.


#### Meta Responses 

>[Documentation](../../docs/source/architecture/agents/intent-classifier.md)

In [ ]:
!.venv/bin/nat run --config_file config_simple_researcher.yml --input "Hello, what can you do?" | sed 's/\\n/\n/g'

#### Shallow Research
>[Documentation](../../docs/source/architecture/agents/shallow-researcher.md)

In [ ]:
!.venv/bin/nat run --config_file config_simple_researcher.yml --input "What is NVIDIA Spectrum-X primarily designed for?" | sed 's/\\n/\n/g'

#### Deep Research 

>[Documentation](../../docs/source/architecture/agents/deep-researcher.md)  
<div class="alert alert-block alert-info">
<b>Note:</b> Deep research can take several minutes: multiple LLM calls and web searches run in sequence. Watch the logs to see planning and research steps.
</div>

In [ ]:
!.venv/bin/nat run --config_file config_simple_researcher.yml --input "Conduct a comprehensive technical comparison between NVIDIA GB300 and Google Ironwood AI accelerator platforms. Generate a structured report with the following sections: Compute Architecture, Memory Subsystem, Interconnect & Scalability, System & Power, Software Ecosystem and Benchmarks" | sed 's/\\n/\n/g'

## (Optional) Add Observability with LangSmith

[LangSmith](https://smith.langchain.com/) lets you trace, monitor, and debug your AI-Q agent runs end-to-end.

**1. Sign up and get an API key**

Go to [smith.langchain.com](https://smith.langchain.com/), create an account, and generate an API key. Create a project to collect your traces.

**2. Set your LangSmith environment variables**

In [ ]:
import getpass
import os

if "LANGSMITH_API_KEY" not in os.environ or os.environ["LANGSMITH_API_KEY"] == "":
    langsmith_api_key = getpass.getpass("Enter your LangSmith API key: ")
    os.environ["LANGSMITH_API_KEY"] = langsmith_api_key

if "LANGSMITH_PROJECT" not in os.environ or os.environ["LANGSMITH_PROJECT"] == "":
    os.environ["LANGSMITH_PROJECT"] = input("Enter your LangSmith project name: ")

**3. Add LangSmith tracing to your config**

Run the cell below to automatically update `config_simple_researcher.yml` with LangSmith tracing:

In [ ]:
config_path = "config_simple_researcher.yml"

with open(config_path) as f:
    content = f.read()

if "tracing:" not in content:
    content = content.replace(
        "        level: INFO\n",
        "        level: INFO\n\n"
        "    tracing:\n"
        "      langsmith:\n"
        "        _type: langsmith\n"
        "        project: ${LANGSMITH_PROJECT}\n"
        "        api_key: ${LANGSMITH_API_KEY}\n",
        1,
    )
    with open(config_path, "w") as f:
        f.write(content)
    print(f"Tracing config added to {config_path}")
else:
    print("Tracing config already present — no changes made.")

**4. Run the agent and verify traces**

Re-run the agent with the updated config:

In [ ]:
!.venv/bin/nat run --config_file config_simple_researcher.yml --input "What is NVIDIA Spectrum-X?" | sed 's/\\n/\n/g'

After the run completes, go to [smith.langchain.com](https://smith.langchain.com/), open your project, and you should see a new trace for the query above. Click into it to explore the agent's steps, LLM calls, and tool usage.

## (Optional) Use a Partner LLM Endpoint

You can swap the LLM provider in `config_simple_researcher.yml` with any OpenAI-compatible API.

For example, to use [Together.ai](https://www.together.ai/) with the [Nemotron model](https://www.together.ai/models/nvidia-nemotron-3-nano) — sign up at [together.ai](https://www.together.ai/), generate an API key, and update the Nemotron LLM blocks (`nemotron_llm_intent`, `nemotron_nano_llm`) following this pattern:

```yaml
llms:
  nemotron_nano_llm:
    _type: openai
    model_name: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
    base_url: "https://api.together.xyz/v1"
    api_key: ${TOGETHER_API_KEY}
    temperature: 0.1
    top_p: 0.3
    max_tokens: 16384
    num_retries: 5
```

Key changes from the default NIM config:
- `_type: nim` → `_type: openai`
- `base_url` → partner endpoint URL
- `api_key` → your `TOGETHER_API_KEY`
- Remove `chat_template_kwargs: enable_thinking: true` (NIM-specific)

> More provider guides coming soon.

## Next Steps

You have now deployed the NVIDIA AI-Q Blueprint and interacted with the chat-based agent. To continue this journey, checkout the [Github repo](https://github.com/NVIDIA-AI-Blueprints/aiq).

**More notebooks**: We have walkthroughs of the deep researcher agent and its customization (see [notebooks](../notebooks/)).

**Customization Guide**: Checkout the [customization guide](../source/customization/)